In [4]:
import pandas as pd
from scipy import stats
from datetime import datetime
now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M")


FEATURES = "linguistics_only"
FEATURES = "linguistics_and_demographics"
#FEATURES = "demographics_only"


# load results for all targets and get latex table
model_results_all_targets = pd.read_csv("between_vs_within_effect.csv").rename(columns={'Unnamed: 0': 'param'}).query(f"feature_set == '{FEATURES}'")
assert model_results_all_targets['n_obs'].std() == 0
assert model_results_all_targets['n_persons'].std() == 0
model_results_all_targets

,param,coef,pval,ci_low,ci_high,target,version,n_obs,n_persons,feature_set
0,Intercept,-1.473148,1.907264e-08,-1.986886,-0.959409,language,mundlak,151,65,linguistics_and_demographics
1,mmse,0.030675,4.670577e-03,0.009423,0.051928,language,mundlak,151,65,linguistics_and_demographics
2,mmse_person_first,-0.002097,8.952597e-01,-0.033315,0.029121,language,mundlak,151,65,linguistics_and_demographics
3,sample_name_longitudinal Var,0.358398,5.467828e-02,-0.007189,0.723985,language,mundlak,151,65,linguistics_and_demographics
4,Intercept,-1.473148,1.907264e-08,-1.986886,-0.959409,language,between-vs-within,151,65,linguistics_and_demographics
5,mmse_deviation_from_person_first,0.030675,4.670577e-03,0.009423,0.051928,language,between-vs-within,151,65,linguistics_and_demographics
6,mmse_person_first,0.028579,1.053578e-02,0.006679,0.050478,language,between-vs-within,151,65,linguistics_and_demographics
7,sample_name_longitudinal Var,0.358398,5.467828e-02,-0.007189,0.723985,language,between-vs-within,151,65,linguistics_and_demographics


In [5]:
Z = stats.norm.ppf(0.975)
WITHIN, PERSON_MEAN = 'mmse_deviation_from_person_first', 'mmse_person_first'

df = model_results_all_targets

def pick(version, param, col='coef'):
    """Value of one coefficient per target, from one parameterization."""
    sub = df[(df['version'] == version) & (df['param'] == param)]
    return sub.set_index('target')[col]

def fmt(coef, lo, hi, p):
    fmt_pval = lambda pval: f"p={pval:.2f}" if pval >= 0.01 else "p<0.01"
    return pd.Series([f"${c:.3f}$ [${l:.3f}$, ${h:.3f}$] ${fmt_pval(pv)}$"
                      for c, l, h, pv in zip(coef, lo, hi, p)], index=coef.index)


model_results_table = pd.DataFrame({
    # within-person: identical in both parameterizations
    'Within-person coefficient':  fmt(pick('between-vs-within', WITHIN),
                          pick('between-vs-within', WITHIN, 'ci_low'),
                          pick('between-vs-within', WITHIN, 'ci_high'),
                          pick('between-vs-within', WITHIN, 'pval')),
    # between-person: person-mean coefficient in the ORTHOGONAL form only
    'Between-person coefficient': fmt(pick('between-vs-within', PERSON_MEAN),
                          pick('between-vs-within', PERSON_MEAN, 'ci_low'),
                          pick('between-vs-within', PERSON_MEAN, 'ci_high'),
                          pick('between-vs-within', PERSON_MEAN, 'pval')),
    # difference (between - within): person-mean coefficient in the MUNDLAK form
    'Difference (between - within)':     fmt(pick('mundlak', PERSON_MEAN),
                          pick('mundlak', PERSON_MEAN, 'ci_low'),
                          pick('mundlak', PERSON_MEAN, 'ci_high'),
                          pick('mundlak', PERSON_MEAN, 'pval')),
})

def rename_target(t):
    return t.replace("_", " ").title().replace("Executive Function", "EF")

model_results_table.index = [rename_target(target) for target in model_results_table.index]
model_results_table

,Within-person coefficient,Between-person coefficient,Difference (between - within)
Language,"$0.031$ [$0.009$, $0.052$] $p<0.01$","$0.029$ [$0.007$, $0.050$] $p=0.01$","$-0.002$ [$-0.033$, $0.029$] $p=0.90$"


In [6]:
print(" %% ", FEATURES, now())
print(model_results_table.to_latex(index=True, header=True, caption=f"\\change{{Results of a linear mixed-effect model applying a model trained on our main dataset to an external clinical dataset of {int(model_results_all_targets['n_persons'].mean())} participants whose MMSE varied across visits. Coefficients give the change in predicted composite score per MMSE point. We report coefficient estimates with 95\\% confidence intervals and corresponding $p$-values.}}", label="tab:pitt_model_results"))

 %%  linguistics_and_demographics 2026-09-22 12:02
\begin{table}
\caption{\change{Results of a linear mixed-effect model applying a model trained on our main dataset to an external clinical dataset of 65 participants whose MMSE varied across visits. Coefficients give the change in predicted composite score per MMSE point. We report coefficient estimates with 95\% confidence intervals and corresponding $p$-values.}}
\label{tab:pitt_model_results}
\begin{tabular}{llll}
\toprule
 & Within-person coefficient & Between-person coefficient & Difference (between - within) \\
\midrule
Language & $0.031$ [$0.009$, $0.052$] $p<0.01$ & $0.029$ [$0.007$, $0.050$] $p=0.01$ & $-0.002$ [$-0.033$, $0.029$] $p=0.90$ \\
\bottomrule
\end{tabular}
\end{table}

